In [ ]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Allow running from either the project root or the test folder.
ROOT = Path.cwd()
if not (ROOT / "src/allocations.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Pick up edits to allocations.py when this cell is rerun.
import src.allocations

importlib.reload(src.allocations)
from src.allocations import Allocations

In [ ]:
method = "test_bootstrap_nn"
# Variables
TEST_SIZE = 50
# 0.1, 0.25, 0.5
SELECTION_RATE = 0.5
# qualified_gagne_1, qualified_gagne_2, qualified_gagne_3
QUALIFICATION_COLUMN = "qualified_gagne_1"

NEED_COLUMN = "gagne_sum_t"
GROUP_COLUMN = "race"
RASHOMON_EPSILON = 0.01
ALLOCATION_DELTA = 0.1
NUM_MODELS = 20
ITERATIONS_PER_SPLIT = 5

fp_input = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
fp_predictions = ROOT / f"data/predictions/{method}_{QUALIFICATION_COLUMN}.csv"
fp_output_opt = ROOT / f"data/results/allocations/opt_tru_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
fp_output_lda = ROOT / f"data/results/allocations/lda_tru_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
fp_output_people = ROOT / f"data/results/allocations/people_tru_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
fp_output = ROOT / f"data/results/comparison_tru_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"

In [ ]:
data = pd.read_csv(fp_input)
df = pd.read_csv(fp_predictions)
df = df.merge(
    data[["person_id", GROUP_COLUMN]],
    on="person_id",
    how="left",
    validate="many_to_one",
)

In [ ]:
a = Allocations(
    df=df,
    TEST_SIZE=TEST_SIZE,
    SELECTION_RATE=SELECTION_RATE,
    QUALIFICATION_COLUMN="y",
    NEED_COLUMN=NEED_COLUMN,
    GROUP_COLUMN=GROUP_COLUMN,
    RASHOMON_EPSILON=RASHOMON_EPSILON,
    ALLOCATION_EPSILON=ALLOCATION_DELTA,
    NUM_MODELS=NUM_MODELS,
    ITERATIONS_PER_SPLIT=ITERATIONS_PER_SPLIT,
)
allocations_opt, allocations_lda, people = a.get_allocations()
print(f"{len(people)} cohorts, {TEST_SIZE} people per cohort, k = {a.k}")

fp_output_opt.parent.mkdir(parents=True, exist_ok=True)
fp_output_lda.parent.mkdir(parents=True, exist_ok=True)
fp_output_people.parent.mkdir(parents=True, exist_ok=True)

allocations_opt.to_csv(fp_output_opt)
allocations_lda.to_csv(fp_output_lda)
people.to_csv(fp_output_people)

## Calculate Utility and Disparity

For each cohort, we look up each allocation's selected people by `person_id` and calculate:

- **Expected utility:** average probability under the model that produced the allocation.
- **Realized utility:** average observed qualification label `y` among selected people.
- **Absolute disparity:** absolute difference between the selected mean needs of groups 1 and 0.
- **Disparity ratio:** selected mean need of group 1 divided by that of group 0.

A missing selected group makes disparity undefined; a zero group-0 mean also makes the ratio undefined. These values are recorded as `NaN`. Our class currently falls back to top-k if DP finds no feasible solution, so a row labeled `DP` can contain that fallback allocation.

In [ ]:
all_results = []

for _, cohort_info in people.iterrows():
    seed = cohort_info["seed"]
    iteration = cohort_info["iteration"]

    split_df = df.loc[(df["seed"] == seed) & (df["person_id"] >= 0)]
    split_df = split_df.set_index("person_id")
    cohort = split_df.loc[cohort_info["people"]]

    for algorithm, allocations in [("OPT", allocations_opt), ("LDA", allocations_lda)]:
        cohort_allocations = allocations.loc[(allocations["seed"] == seed) & (allocations["iteration"] == iteration)]

        for _, allocation in cohort_allocations.iterrows():
            selected = cohort.loc[allocation["selected"]]

            mean_0 = selected.loc[selected[GROUP_COLUMN] == 0, NEED_COLUMN].mean()
            mean_1 = selected.loc[selected[GROUP_COLUMN] == 1, NEED_COLUMN].mean()
            absolute_disparity = abs(mean_1 - mean_0)
            disparity_ratio = np.nan
            if pd.notna(mean_0) and pd.notna(mean_1) and mean_0 != 0:
                disparity_ratio = mean_1 / mean_0

            for model in allocation["models"]:
                all_results.append({
                    "method": method,
                    "algorithm": algorithm,
                    "qualification_column": QUALIFICATION_COLUMN,
                    "seed": seed,
                    "iteration": iteration,
                    "model": model,
                    "k": a.k,
                    "epsilon": ALLOCATION_DELTA,
                    "tolerance_type": "add",
                    "expected_utility": selected[model].mean(),
                    "realized_utility": selected["y"].mean(),
                    "absolute_disparity": absolute_disparity,
                    "disparity_ratio": disparity_ratio,
                })

results = pd.DataFrame(all_results)
results

In [ ]:
results.groupby(["seed", "iteration", "method", "qualification_column", "algorithm"]).mean(True)

In [ ]:
fp_output.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(fp_output, index=False)
print(f"Saved {len(results)} rows to {fp_output}")